# Sign Language Detection — Part 1: Data Collection


| Setting | Value |
|---|---|
| Sequences per gesture | 60 |
| Frames per sequence | 40 |
| Keypoint vector size | 258 |

## 1. Imports

In [2]:
import cv2
import numpy as np
import os
import mediapipe as mp

## 2. Configuration


In [3]:
WORD            = "Fail"  
DATA_PATH       = "MP_Data"
NO_SEQUENCES    = 60
START_SEQUENCE  = 0         
SEQUENCE_LENGTH = 40
INPUT_SIZE      = 258       

actions = np.array([WORD])
print(f"Collecting  : {WORD}")
print(f"Sequences   : {NO_SEQUENCES}")
print(f"Frames/seq  : {SEQUENCE_LENGTH}")


Sequences   : 60
Frames/seq  : 40


## 3. Create Directory Structure

In [4]:
for action in actions:
    for seq in range(START_SEQUENCE, START_SEQUENCE + NO_SEQUENCES):
        os.makedirs(os.path.join(DATA_PATH, action, str(seq)), exist_ok=True)

print(f"Directories ready under: {DATA_PATH}/{WORD}/")


Directories ready under: MP_Data/Fail/


## 4. MediaPipe Helper Functions

In [5]:
mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils


def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = model.process(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results


def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks,  mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)


def extract_keypoints(results):
  
    pose = (
        np.array([[r.x, r.y, r.z, r.visibility] for r in results.pose_landmarks.landmark]).flatten()
        if results.pose_landmarks else np.zeros(33 * 4)
    )
    lh = (
        np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten()
        if results.left_hand_landmarks else np.zeros(21 * 3)
    )
    rh = (
        np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten()
        if results.right_hand_landmarks else np.zeros(21 * 3)
    )
    return np.concatenate([pose, lh, rh])


## 5. Collect Data

- A **1.2-second countdown** appears before each sequence.
- Press **`q`** to stop early.

In [6]:
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

with mp_holistic.Holistic(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    for action in actions:
        for seq in range(START_SEQUENCE, START_SEQUENCE + NO_SEQUENCES):

            # Countdown frame
            ret, frame = cap.read()
            frame = cv2.flip(frame, 1)
            cv2.putText(frame, f"GET READY: {action}  seq {seq+1}/{NO_SEQUENCES}",
                        (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
            cv2.imshow("Collection", frame)
            cv2.waitKey(1200)

            # Capture frames
            for frame_num in range(SEQUENCE_LENGTH):
                ret, frame = cap.read()
                frame = cv2.flip(frame, 1)
                image, results = mediapipe_detection(frame, holistic)
                draw_landmarks(image, results)

                cv2.putText(image, f"WORD: {action}",                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(image, f"SEQ: {seq+1}/{NO_SEQUENCES}",           (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(image, f"FRAME: {frame_num+1}/{SEQUENCE_LENGTH}",(10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

                keypoints = extract_keypoints(results)
                np.save(os.path.join(DATA_PATH, action, str(seq), str(frame_num)), keypoints)

                cv2.imshow("Collection", image)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

cap.release()
cv2.destroyAllWindows()
print(f"Done collecting: {WORD}")


c:\DLPROJECTTTT\Realtime-Sign-Language-Detection-Using-LSTM-Model\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Done collecting: Fail


## 6. Verify Collected Data

Run this after collection to confirm all sequences are complete and correct.

In [ ]:
print(f"Verifying: {WORD}\n")
action_path = os.path.join(DATA_PATH, WORD)
all_ok = True

for seq in sorted(os.listdir(action_path), key=lambda x: int(x)):
    seq_path = os.path.join(action_path, seq)
    files    = sorted(os.listdir(seq_path), key=lambda x: int(x.split(".")[0]))

    if len(files) != SEQUENCE_LENGTH:
        print(f"  [WARN] seq {seq}: {len(files)}/{SEQUENCE_LENGTH} frames")
        all_ok = False
        continue

    sample = np.load(os.path.join(seq_path, files[0]))
    if sample.shape[0] != INPUT_SIZE:
        print(f"  [WARN] seq {seq}: shape mismatch {sample.shape} (expected ({INPUT_SIZE},))")
        all_ok = False

if all_ok:
    print(f"  [OK] All {NO_SEQUENCES} sequences verified for '{WORD}'")
else:
    print("\n  Re-collect the flagged sequences before training.")
